In [3]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch
from mas_execution_manager.scenario_state_base import ScenarioStateBase
from mdr_listen_action.msg import ListenAction, ListenGoal
class receptionist():
    def __init__(self):
        # ... your existing initialization code ...
        
        # Load Hugging Face Model and Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")
        self.model = AutoModelForTokenClassification.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")

    def extract_guest_info(self, user_input):
        inputs = self.tokenizer(user_input, return_tensors="pt")
        outputs = self.model(**inputs).logits
        predictions = torch.argmax(outputs, dim=2)

        tokens = inputs.tokens()
        labels = [self.model.config.id2label[prediction] for prediction in predictions[0].numpy()]
        
        guest_name = None
        favorite_drink = None  # You might need a custom method to identify drinks, as this model does not directly support it

        for token, label in zip(tokens, labels):
            if label == "B-PER":
                guest_name = token.strip("[CLS]").strip("[SEP]")  # Adjust extraction logic as needed
            # Implement custom logic for favorite drink if possible

        return guest_name, favorite_drink


In [4]:
talker=receptionist()

In [6]:
talker.extract_guest_info("My name is Ron and I like eating")

(None, None)

In [7]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

nlp = pipeline("ner", model=model, tokenizer=tokenizer)
example = "My name is Wolfgang and I live in Berlin"

ner_results = nlp(example)
print(ner_results)

2024-03-26 16:53:22.568323: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-26 16:53:25.565959: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


[{'entity': 'B-PER', 'score': 0.9990139, 'index': 4, 'word': 'Wolfgang', 'start': 11, 'end': 19}, {'entity': 'B-LOC', 'score': 0.999645, 'index': 9, 'word': 'Berlin', 'start': 34, 'end': 40}]


In [8]:
nlp = pipeline("ner", model=model, tokenizer=tokenizer)
example = "My name is Wolfgang and I live in Berlin"

ner_results = nlp(example)
print(ner_results)

[{'entity': 'B-PER', 'score': 0.9990139, 'index': 4, 'word': 'Wolfgang', 'start': 11, 'end': 19}, {'entity': 'B-LOC', 'score': 0.999645, 'index': 9, 'word': 'Berlin', 'start': 34, 'end': 40}]


In [10]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Initialize tokenizer and model from the pretrained versions
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

# Initialize NER pipeline
nlp = pipeline("ner", model=model, tokenizer=tokenizer)

# Example sentence


# Function to extract entities based on label
def extract_entities(ner_results, label):
    return [entity['word'] for entity in ner_results if entity['entity'] == label]

# Known drinks list for this example (you might want to expand this list)

known_drinks = [
    'Espresso', 'Coffee', 'Tea', 'Water', 'Latte', 'Cappuccino', 'Mocha', 'Americano', 'Macchiato',
    'Flat White', 'Chai Latte', 'Green Tea', 'Black Tea', 'Herbal Tea', 'Oolong Tea', 'White Tea',
    'Matcha', 'Hot Chocolate', 'Milk', 'Soy Milk', 'Almond Milk', 'Oat Milk', 'Coconut Milk',
    'Lemonade', 'Orange Juice', 'Apple Juice', 'Grape Juice', 'Tomato Juice', 'Pineapple Juice',
    'Cranberry Juice', 'Mango Juice', 'Beet Juice', 'Carrot Juice', 'Ginger Beer', 'Tonic Water',
    'Soda Water', 'Coca Cola', 'Pepsi', 'Sprite', 'Fanta', '7 Up', 'Root Beer', 'Ginger Ale',
    'Red Wine', 'White Wine', 'Beer', 'Ale', 'Lager', 'Cider', 'Whiskey', 'Vodka', 'Rum', 'Gin',
    'Tequila', 'Brandy', 'Sake', 'Champagne', 'Prosecco', 'Martini', 'Margarita', 'Mojito',
    'Sangria', 'Cosmopolitan', 'Old Fashioned', 'Manhattan', 'Negroni', 'Bloody Mary', 'Mimosa',
    'Pina Colada', 'Daiquiri', 'Long Island Iced Tea', 'Sex on the Beach', 'Caipirinha', 'Mai Tai',
    'Moscow Mule', 'French 75', 'Gin and Tonic', 'Rum Punch', 'Whiskey Sour', 'Bellini', 'Kir Royale'
]

# Extract persons


Persons: ['Wolfgang', 'E']
Drinks: ['Espresso']


In [11]:
example = "My name is Zain, and I love drinking Cola."

# Process the example sentence through the NER pipeline
ner_results = nlp(example)

persons = extract_entities(ner_results, 'B-PER')

# Attempt to extract drinks by finding known drink names in the sentence
# This simplistic approach checks if any known drink is mentioned in the NER results.
# In a more robust application, you might use additional context or a dedicated classifier.
drinks = [word for word in known_drinks if word in example]

print(f"Persons: {persons}")
print(f"Drinks: {drinks}")


Persons: ['Z']
Drinks: []


In [21]:
def aggregate_entities(ner_results):
    """
    Aggregate tokens belonging to multi-token entities.
    """
    aggregated_entities = {'PERSON': [], 'DRINK': []}
    current_entity = None

    for entity in ner_results:
        # Corrected key for entity type
        entity_type = entity['entity']
        if 'PER' in entity_type:
            word = entity['word'].replace("##", "")
            if current_entity:
                current_entity += ' ' + word
            else:
                current_entity = word
        else:
            if current_entity:
                aggregated_entities['PERSON'].append(current_entity)
                current_entity = None

    # Add the last entity if it exists
    if current_entity:
        aggregated_entities['PERSON'].append(current_entity)

    # Attempt to extract drinks by finding known drink names in the ner results
    for word in known_drinks:
        if word.lower() in example.lower():
            aggregated_entities['DRINK'].append(word)

    return aggregated_entities

# Known drinks list for this example (you might want to expand this list)
known_drinks = ['Espresso', 'Coffee', 'Tea', 'Water', 'Latte', 'Cappuccino']

# Process the example sentence through the NER pipeline
ner_results = nlp("My name is Zain and i like Coke")

# Aggregate entities
entities = aggregate_entities(ner_results)

print(f"Persons: {entities['PERSON']}")
print(f"Drinks: {entities['DRINK']}")


Persons: ['Z ain Coke']
Drinks: ['Espresso']


In [22]:
import polyglot
from polyglot.downloader import downloader
print(downloader.supported_languages_table("ner2", 3))
downloader.download("embeddings2.en")
downloader.download("ner2.en")


  1. Italian                    2. Hindi                      3. French                   
  4. Spanish; Castilian         5. Vietnamese                 6. Arabic                   
  7. Bulgarian                  8. Norwegian                  9. Estonian                 
 10. Japanese                  11. Greek, Modern             12. Slovene                  
 13. Korean                    14. Serbian                   15. Finnish                  
 16. Catalan; Valencian        17. Croatian                  18. Dutch                    
 19. Swedish                   20. Tagalog                   21. Danish                   
 22. Latvian                   23. Ukrainian                 24. Romanian, Moldavian, ... 
 25. Persian                   26. Slovak                    27. Portuguese               
 28. English                   29. Malay                     30. Polish                   
 31. German                    32. Indonesian                33. Chinese                  

True

In [23]:
from polyglot.text import Text

# Predefined list of known drinks
known_drinks = ['espresso', 'coffee', 'tea', 'water', 'latte', 'cappuccino', 'mocha', 'americano', 'macchiato', 'flat white', 'chai latte', 'green tea', 'black tea', 'herbal tea', 'oolong tea', 'white tea', 'matcha', 'hot chocolate', 'milk', 'lemonade', 'orange juice', 'apple juice', 'grape juice', 'tomato juice', 'pineapple juice', 'cranberry juice', 'mango juice', 'soda', 'cola', 'beer', 'wine', 'champagne', 'whiskey', 'vodka', 'rum', 'gin', 'tequila']

# Example sentence
example = "My name is Wolfgang and I love drinking espresso."

# Process the example sentence
text = Text(example)
entities = text.entities

# Extract person names
persons = [entity for entity in entities if entity.tag == 'I-PER']

# Attempt to extract drinks by checking if any known drink name is in the sentence
drinks = [drink for drink in known_drinks if drink in example.lower()]

print(f"Persons: {persons}")
print(f"Drinks: {drinks}")


Persons: [I-PER(['Wolfgang'])]
Drinks: ['espresso']


In [45]:
import polyglot
from polyglot.text import Text



# Predefined list of known drinks
known_drinks = [
    'Espresso', 'Coffee', 'Tea', 'Water', 'Latte', 'Cappuccino', 'Mocha', 'Americano', 'Macchiato',
    'Flat White', 'Chai Latte', 'Green Tea', 'Black Tea', 'Herbal Tea', 'Oolong Tea', 'White Tea',
    'Matcha', 'Hot Chocolate', 'Milk', 'Soy Milk', 'Almond Milk', 'Oat Milk', 'Coconut Milk',
    'Lemonade', 'Orange Juice', 'Apple Juice', 'Grape Juice', 'Tomato Juice', 'Pineapple Juice',
    'Cranberry Juice', 'Mango Juice', 'Beet Juice', 'Carrot Juice', 'Ginger Beer', 'Tonic Water',
    'Soda Water', 'Coca Cola', 'Pepsi', 'Sprite', 'Fanta', '7 Up', 'Root Beer', 'Ginger Ale',
    'Red Wine', 'White Wine', 'Beer', 'Ale', 'Lager', 'Cider', 'Whiskey', 'Vodka', 'Rum', 'Gin',
    'Tequila', 'Brandy', 'Sake', 'Champagne', 'Prosecco', 'Martini', 'Margarita', 'Mojito',
    'Sangria', 'Cosmopolitan', 'Old Fashioned', 'Manhattan', 'Negroni', 'Bloody Mary', 'Mimosa',
    'Pina Colada', 'Daiquiri', 'Long Island Iced Tea', 'Sex on the Beach', 'Caipirinha', 'Mai Tai',
    'Moscow Mule', 'French 75', 'Gin and Tonic', 'Rum Punch', 'Whiskey Sour', 'Bellini', 'Kir Royale',
    # Adding 'Cola' to cover common requests like 'cola'
    'Cola'
]

# Example sentence
example_sentence = "I am Zain and I love drinking tea."

# Analyze the sentence using Polyglot
text = Text(example_sentence, hint_language_code='en')

# Extract person names
persons = [entity for entity in text.entities if entity.tag == 'I-PER']
persons_names = [entity[0] for entity in persons]
# Lowercase the known_drinks for case-insensitive matching
known_drinks_lower = [drink.lower() for drink in known_drinks]

# Normalize the sentence to lowercase
sentence_lower = example_sentence.lower()

# Identify drinks based on the known drinks list, matching the whole or part of the drink name in the sentence
matched_drinks = [drink for drink in known_drinks if drink.lower() in sentence_lower]

print(f"Persons: {persons_names}")
print(f"Drinks: {matched_drinks}")


Persons: ['Zain']
Drinks: ['Tea']


: 